In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math

In [3]:
class InputEmbedding(nn.Module):

  def __init__(self, d_model, vocab_size):
    super().__init__()
    self.d_model = d_model
    self.vocab_size = vocab_size
    self.embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)

  def forward(self, x):

    return self.embed(x) * math.sqrt(self.d_model)


In [4]:
class PositionalEncoding(nn.Module):

  def __init__(self, d_model, seq_len, dropout):
    super().__init__()
    self.d_model = d_model
    self.seq_len = seq_len
    self.dropout = nn.Dropout(dropout)
    pe = torch.zeros(seq_len, d_model) # matrix of shape same as embedings
    pos = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1) # tensor of shape [seq_len, 1] denotes the position of token
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # shape of tensor div_term = [d_model // 2]
    pe[:, 0::2] = torch.sin(pos * div_term)
    pe[:, 1::2] = torch.cos(pos * div_term)
    pe = pe.unsqueeze(0) # shape of pe = [1, seq_len, d_model]

    self.register_buffer('pe', pe)

  def forward(self, x):
    x = x + self.pe[:, :x.shape[1], :].requires_grad_(False)  # slicing is done to avoid shape mismatch in variable length sequence
    return self.dropout(x)

In [5]:
class LayerNorm(nn.Module):

  def __init__(self, d_model, epsilon = 10**-6):

    super().__init__()
    self.epsilon = epsilon
    self.gamma = nn.Parameter(torch.ones(d_model))
    self.beta = nn.Parameter(torch.zeros(d_model))

  # x shape = [batch_size, seq_len, d_model]
  def forward(self, x):

    mean = x.mean(dim=-1, keepdim=True)
    std = x.std(dim=-1, keepdim=True)

    return self.gamma * (x - mean) / (std + self.epsilon) + self.beta # mathematically not exact

In [6]:
class FeedForward(nn.Module):

  def __init__(self, d_model, d_ff, dropout):

    super().__init__()
    self.layer1 = nn.Linear(d_model, d_ff)
    self.layer2 = nn.Linear(d_ff, d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):

    return self.layer2(self.dropout(torch.relu(self.layer1(x))))

In [7]:
class MHA(nn.Module):

  def __init__(self, d_model, h, dropout):

    super().__init__()
    self.d_model = d_model
    self.h = h
    self.dropout = nn.Dropout(dropout)

    self.d_k = d_model // h # d_k = d_v
    self.w_q = nn.Linear(d_model, d_model)
    self.w_k = nn.Linear(d_model, d_model)
    self.w_v = nn.Linear(d_model, d_model)

    self.w_o = nn.Linear(d_model, d_model)

  def forward(self, q, k, v, mask):

    batch_size, seq_len, _ = q.size()

    query = self.w_q(q) # shape of both query and key = [batch_size, seq_len, d_model]
    key = self.w_k(k) # same as query
    value = self.w_v(v) # same as query

    query = query.view(batch_size, -1, self.h, self.d_k) # shape = [batch_size, seq_len, h, d_k]
    query = query.transpose(1, 2) # shape = [batch_size, h, seq_len, d_k]
    key = key.view(batch_size, -1, self.h, self.d_k)
    key = key.transpose(1, 2)
    value = value.view(batch_size, -1, self.h, self.d_k)
    value = value.transpose(1, 2)

    attention_scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(self.d_k) # shape = [batch_size, h, seq_len, seq_len]

    if mask is not None:
      attention_scores = attention_scores.masked_fill_(mask == 0, float('-inf'))

    attention_weights = attention_scores.softmax(dim=-1)

    if self.dropout is not None:
      attention_weights = self.dropout(attention_weights)

    attention_output = attention_weights @  value # shape = [batch_size, h, seq_len, d_k]

    attention_output = attention_output.transpose(1, 2) # shape = [batch_size, seq_len, h, d_k]
    attention_output = attention_output.contiguous() # makes the tensor contiguous in memory for .view as transpose may result in tensor not being stored in a contiguous block of memory
    attention_output = attention_output.view(batch_size, seq_len, self.d_model) # shape = [batch_size, seq_len, d_model]
    attention_output = self.w_o(attention_output) # final projection, same shape
    return attention_output

In [8]:
class SkipConnection(nn.Module):

  def __init__(self, dropout, d_model):

    super().__init__()
    self.dropout = nn.Dropout(dropout)
    self.norm = LayerNorm(d_model)

  def forward(self, x, sublayer):

    return x + self.dropout(sublayer(self.norm(x))) # pre-norm

In [9]:
class EncoderBlock(nn.Module):

  def __init__(self, attention, ffn, dropout, d_model):

    super().__init__()
    self.attention = attention
    self.ffn = ffn
    self.residual = nn.ModuleList([SkipConnection(dropout, d_model) for _ in range(2)])

  # src_mask is used to mask out padding tokens in encoder
  def forward(self, x, src_mask):
    x = self.residual[0](x, lambda y: self.attention(y, y, y, src_mask))
    x = self.residual[1](x, self.ffn)
    return x

In [10]:
class Encoder(nn.Module):

  def __init__(self, d_model, layers):

    super().__init__()
    self.layers = layers
    self.norm = LayerNorm(d_model)

  def forward(self, x, mask):

    for layer in self.layers:
      x = layer(x, mask)
    return self.norm(x)

In [11]:
class DecoderBlock(nn.Module):

  def __init__(self, self_attention, cross_attention, ffn, dropout, d_model):

    super().__init__()
    self.self_attention = self_attention
    self.cross_attention = cross_attention
    self.ffn = ffn
    self.residual = nn.ModuleList([SkipConnection(dropout, d_model) for _ in range(3)])

  def forward(self, x, encoder_output, src_mask, trg_mask):

    x = self.residual[0](x, lambda y: self.self_attention(y, y, y, trg_mask))
    x = self.residual[1](x, lambda y: self.cross_attention(y, encoder_output, encoder_output, src_mask))
    x = self.residual[2](x, self.ffn)

    return x

In [12]:
class Decoder(nn.Module):

  def __init__(self, d_model, layers):

    super().__init__()
    self.layers = layers
    self.norm = LayerNorm(d_model)

  def forward(self, x, encoder_output, src_mask, trg_mask):

    for layer in self.layers:
      x = layer(x, encoder_output, src_mask, trg_mask)

    return self.norm(x)

In [13]:
class Output(nn.Module):

  def __init__(self, d_model, vocab_size):

    super().__init__()
    self.proj = nn.Linear(d_model, vocab_size)

  def forward(self, x):

    return self.proj(x)

In [14]:
class Transformer(nn.Module):

  def __init__(self, encoder, decoder, src_embed, trg_embed, src_pos, trg_pos, output):

    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.src_embed = src_embed
    self.trg_embed = trg_embed
    self.src_pos = src_pos
    self.trg_pos = trg_pos
    self.output_layer = output

  def encode(self, src, src_mask):

    src = self.src_embed(src)
    src = self.src_pos(src)
    return self.encoder(src, src_mask)

  def decode(self, encoder_output, src_mask, trg, trg_mask):

    trg = self.trg_embed(trg)
    trg = self.trg_pos(trg)
    return self.decoder(trg, encoder_output, src_mask, trg_mask)

  def project(self, x):

    return self.output_layer(x)

  def forward(self, src, trg):
        # Create masks for source and target
        # Target mask is a combination of padding mask and subsequent mask
        src_mask = (src != PAD_token).unsqueeze(1).unsqueeze(2) # (batch, 1, 1, src_len)
        trg_mask = (trg != PAD_token).unsqueeze(1).unsqueeze(2) # (batch, 1, 1, trg_len)

        seq_length = trg.size(1)
        subsequent_mask = torch.tril(torch.ones(1, seq_length, seq_length)).to(device) # (1, trg_len, trg_len)
        trg_mask = trg_mask & (subsequent_mask==1)

        encoder_output = self.encode(src, src_mask)
        decoder_output = self.decode(encoder_output, src_mask, trg, trg_mask)
        return self.project(decoder_output)

In [15]:
def BuildTransformer(src_vocab_size, trg_vocab_size, src_seq_len, trg_seq_len, d_model=512, N=6, h=8, dropout=0.1, d_ff=2048):

  src_embed = InputEmbedding(d_model, src_vocab_size)
  trg_embed = InputEmbedding(d_model, trg_vocab_size)

  src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
  trg_pos = PositionalEncoding(d_model, trg_seq_len, dropout)

  encoder_blocks = []
  for _ in range(N):
    encoder_self_attention = MHA(d_model, h, dropout)
    ffn = FeedForward(d_model, d_ff, dropout)
    encoder_block = EncoderBlock(encoder_self_attention, ffn, dropout, d_model)
    encoder_blocks.append(encoder_block)

  decoder_blocks = []
  for _ in range(N):
    decoder_mask_attention = MHA(d_model, h, dropout)
    cross_attention = MHA(d_model, h, dropout)
    ffn = FeedForward(d_model, d_ff, dropout)
    decoder_block = DecoderBlock(decoder_mask_attention, cross_attention, ffn, dropout, d_model)
    decoder_blocks.append(decoder_block)

  encoder = Encoder(d_model, nn.ModuleList(encoder_blocks))
  decoder = Decoder(d_model, nn.ModuleList(decoder_blocks))

  projection = Output(d_model, trg_vocab_size)

  transformer = Transformer(encoder, decoder, src_embed, trg_embed, src_pos, trg_pos, projection)

  for p in transformer.parameters():
    if p.dim() > 1:
      nn.init.xavier_uniform_(p)

  return transformer

In [ ]:
! pip install -U datasets huggingface_hub fsspec

In [ ]:
from datasets import load_dataset

MAX_LEN = 512

def filter_long_example(example):
  return len(example['translation']['en']) < MAX_LEN and \
         len(example['translation']['hi']) < MAX_LEN

ds = load_dataset("cfilt/iitb-english-hindi")

ds = ds.filter(filter_long_example)

In [18]:
from tqdm import tqdm
import random

In [19]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

tokenizer.pre_tokenizer = Whitespace()

def get_trainig_corpus(): # iterator for data
  for i in tqdm(range(len(ds["train"]))):
    yield ds["train"][i]["translation"]["en"]
    yield ds["train"][i]["translation"]["hi"]

trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"], vocab_size=30000)

tokenizer.train_from_iterator(get_trainig_corpus(), trainer=trainer)

tokenizer.save("hindi-english_bpe_tokenizer.json")

100%|██████████| 1650500/1650500 [05:12<00:00, 5279.34it/s]


In [20]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("hindi-english_bpe_tokenizer.json")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Add special tokens
if tokenizer.token_to_id("[SOS]") is None:
    tokenizer.add_special_tokens(['[SOS]'])
if tokenizer.token_to_id("[EOS]") is None:
    tokenizer.add_special_tokens(['[EOS]'])

# Define special token IDs
SOS_token = tokenizer.token_to_id('[SOS]')
EOS_token = tokenizer.token_to_id('[EOS]')
PAD_token = tokenizer.token_to_id('[PAD]')

class TranslationDataset(Dataset):
    def __init__(self, dataset, tokenizer, src_lang="en", tgt_lang="hi"):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        src_text = self.dataset[idx]["translation"][self.src_lang]
        tgt_text = self.dataset[idx]["translation"][self.tgt_lang]

        # Tokenize and add special tokens
        src_ids = [SOS_token] + self.tokenizer.encode(src_text).ids + [EOS_token]
        tgt_ids = [SOS_token] + self.tokenizer.encode(tgt_text).ids + [EOS_token]

        return torch.tensor(src_ids), torch.tensor(tgt_ids)

# Create instances of the dataset for training and validation
train_dataset = TranslationDataset(ds["train"], tokenizer)
val_dataset = TranslationDataset(ds["validation"], tokenizer)

def collate_fn(batch):
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        src_batch.append(src_sample)
        tgt_batch.append(tgt_sample)

    # Pad sequences to the length of the longest sequence in the batch
    src_batch = pad_sequence(src_batch, padding_value=PAD_token, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_token, batch_first=True)

    return src_batch, tgt_batch

# Create the DataLoaders
batch_size = 32
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [22]:
config = {
    "epochs": 15,
    "batch_size": 16,
    "d_model": 512,
    "num_layers": 6,
    "num_heads": 8,
    "d_ff": 2048,
    "dropout": 0.1,
    "learning_rate": 0.0001,
    "max_seq_len": 512,
    "accumulation_steps": 4,
    "logging_interval": 200
}

In [23]:
src_vocab_size = tokenizer.get_vocab_size()
trg_vocab_size = tokenizer.get_vocab_size()

src_seq_len = 512
trg_seq_len = 512
d_model = 256
num_heads = 8
num_layers = 6
d_ff = 2048
dropout = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAD_token = tokenizer.token_to_id('[PAD]')

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
model = BuildTransformer(src_vocab_size,
                         trg_vocab_size,
                         src_seq_len,
                         trg_seq_len,
                         d_model,
                         num_layers,
                         num_heads,
                         dropout,
                         d_ff).to(device)

In [26]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_token)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

In [27]:
from torch.cuda.amp import GradScaler, autocast
import time

In [ ]:
scaler = GradScaler()

def train_epoch(model, dataloader, optimizer, criterion, device, config, epoch_num):

    model.train()

    epoch_start_time = time.time()
    total_epoch_loss = 0.0

    chunk_start_time = time.time()
    chunk_losses = []
    chunk_tokens = 0

    num_batches = len(dataloader)

    optimizer.zero_grad()

    for i, (src_batch, tgt_batch) in enumerate(dataloader):
        src_batch = src_batch.to(device)
        tgt_batch = tgt_batch.to(device)

        tgt_input = tgt_batch[:, :-1]
        tgt_out = tgt_batch[:, 1:]

        with torch.cuda.amp.autocast():
            output = model(src_batch, tgt_input)
            output_flat = output.contiguous().view(-1, output.shape[-1])
            tgt_out_flat = tgt_out.contiguous().view(-1)
            loss = criterion(output_flat, tgt_out_flat)
            loss = loss / config['accumulation_steps']

        scaler.scale(loss).backward()

        if (i + 1) % config['accumulation_steps'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        current_loss = loss.item() * config['accumulation_steps']
        chunk_losses.append(current_loss)
        total_epoch_loss += current_loss

        num_target_tokens = (tgt_batch != PAD_token).sum().item()
        chunk_tokens += num_target_tokens

        if (i + 1) % config['logging_interval'] == 0:
            avg_chunk_loss = sum(chunk_losses) / len(chunk_losses)
            chunk_ppl = math.exp(avg_chunk_loss)

            time_for_chunk = time.time() - chunk_start_time
            bps = chunk_tokens / time_for_chunk

            print(f"  Batch {i+1:5d}/{num_batches:5d}   | Avg Loss (last {config['logging_interval']}): {avg_chunk_loss:.4f} | "
                  f"PPL: {chunk_ppl:7.2f} | Tokens/Sec: {bps:7.2f}")

            chunk_start_time = time.time()
            chunk_losses = []
            chunk_tokens = 0

    epoch_duration = time.time() - epoch_start_time
    avg_epoch_loss = total_epoch_loss / num_batches

    return avg_epoch_loss, epoch_duration

/tmp/ipython-input-28-392631358.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [32]:
import math
from tqdm import tqdm
import torch

def evaluate(model, dataloader, criterion, device):

    model.eval()
    total_loss = 0

    with torch.no_grad():
        data_iterator = tqdm(dataloader, desc="Validating")
        for src_batch, tgt_batch in data_iterator:
            src_batch = src_batch.to(device)
            tgt_batch = tgt_batch.to(device)

            tgt_input = tgt_batch[:, :-1]
            tgt_out = tgt_batch[:, 1:]

            output = model(src_batch, tgt_input)
            output_flat = output.contiguous().view(-1, output.shape[-1])
            tgt_out_flat = tgt_out.contiguous().view(-1)

            loss = criterion(output_flat, tgt_out_flat)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [33]:
check_point_path = '/content/drive/MyDrive/transformer_checkpoint.pth'

In [34]:
checkpoint = torch.load(check_point_path, map_location=device)

In [35]:
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scaler.load_state_dict(checkpoint['scaler_state_dict'])

In [36]:
start_epoch = checkpoint['epoch'] + 1

In [37]:
print(start_epoch)

1


In [38]:
if 'loss' in checkpoint:
        best_val_loss = checkpoint['loss']

In [42]:
print(best_val_loss)
print('hello')

4.239692687988281
hello


In [43]:
model_save_path = '/content/drive/MyDrive/transformer_new_checkpoint.pth'

In [44]:
for epoch in range(start_epoch ,config['epochs']):
    print(f"--- Epoch {epoch+1:02d}/{config['epochs']:02d} ---")

    train_loss, train_duration = train_epoch(
        model, train_dataloader, optimizer, criterion, device, config, epoch + 1
    )

    val_loss = evaluate(
        model, val_dataloader, criterion, device
    )

    train_ppl = math.exp(train_loss)
    val_ppl = math.exp(val_loss)

    mins, secs = divmod(train_duration, 60)

    print(f"End of Epoch: {epoch+1:02d} | Time: {int(mins)}m {int(secs)}s")
    print(f"\tEpoch Train Loss: {train_loss:.3f} | Epoch Train PPL: {train_ppl:7.3f}")
    print(f"\tEpoch Val. Loss: {val_loss:.3f} |  Epoch Val. PPL: {val_ppl:7.3f}")
    print("-" * 70)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'loss': val_loss,
    }, model_save_path)

print("Training finished.")


--- Epoch 02/15 ---


/tmp/ipython-input-28-392631358.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Batch   200/51579   | Avg Loss (last 200): 3.6542 | PPL:   38.64 | Tokens/Sec: 5833.98
  Batch   400/51579   | Avg Loss (last 200): 3.6516 | PPL:   38.54 | Tokens/Sec: 5555.78
  Batch   600/51579   | Avg Loss (last 200): 3.6519 | PPL:   38.55 | Tokens/Sec: 5962.92
  Batch   800/51579   | Avg Loss (last 200): 3.6079 | PPL:   36.89 | Tokens/Sec: 5921.04
  Batch  1000/51579   | Avg Loss (last 200): 3.6385 | PPL:   38.03 | Tokens/Sec: 5834.51
  Batch  1200/51579   | Avg Loss (last 200): 3.6229 | PPL:   37.45 | Tokens/Sec: 6047.36
  Batch  1400/51579   | Avg Loss (last 200): 3.6251 | PPL:   37.53 | Tokens/Sec: 5692.06
  Batch  1600/51579   | Avg Loss (last 200): 3.6253 | PPL:   37.54 | Tokens/Sec: 5888.95
  Batch  1800/51579   | Avg Loss (last 200): 3.6319 | PPL:   37.78 | Tokens/Sec: 5728.10
  Batch  2000/51579   | Avg Loss (last 200): 3.6239 | PPL:   37.48 | Tokens/Sec: 5756.35
  Batch  2200/51579   | Avg Loss (last 200): 3.6370 | PPL:   37.98 | Tokens/Sec: 5994.45
  Batch  2400/51579  

Validating: 100%|██████████| 17/17 [00:00<00:00, 20.02it/s]


End of Epoch: 02 | Time: 89m 45s
	Epoch Train Loss: 3.337 | Epoch Train PPL:  28.130
	Epoch Val. Loss: 3.588 |  Epoch Val. PPL:  36.176
----------------------------------------------------------------------
--- Epoch 03/15 ---
  Batch   200/51579   | Avg Loss (last 200): 3.0167 | PPL:   20.42 | Tokens/Sec: 5625.15
  Batch   400/51579   | Avg Loss (last 200): 3.0669 | PPL:   21.48 | Tokens/Sec: 5894.69
  Batch   600/51579   | Avg Loss (last 200): 3.0533 | PPL:   21.18 | Tokens/Sec: 5986.64
  Batch   800/51579   | Avg Loss (last 200): 3.0183 | PPL:   20.46 | Tokens/Sec: 5792.95
  Batch  1000/51579   | Avg Loss (last 200): 3.0279 | PPL:   20.65 | Tokens/Sec: 6048.78
  Batch  1200/51579   | Avg Loss (last 200): 3.0204 | PPL:   20.50 | Tokens/Sec: 5929.95
  Batch  1400/51579   | Avg Loss (last 200): 3.0200 | PPL:   20.49 | Tokens/Sec: 5803.45
  Batch  1600/51579   | Avg Loss (last 200): 3.0542 | PPL:   21.20 | Tokens/Sec: 6055.34
  Batch  1800/51579   | Avg Loss (last 200): 3.0350 | PPL:   

Validating: 100%|██████████| 17/17 [00:00<00:00, 18.61it/s]


End of Epoch: 03 | Time: 90m 21s
	Epoch Train Loss: 2.897 | Epoch Train PPL:  18.116
	Epoch Val. Loss: 3.291 |  Epoch Val. PPL:  26.865
----------------------------------------------------------------------
--- Epoch 04/15 ---
  Batch   200/51579   | Avg Loss (last 200): 2.7296 | PPL:   15.33 | Tokens/Sec: 5056.23
  Batch   400/51579   | Avg Loss (last 200): 2.7104 | PPL:   15.04 | Tokens/Sec: 5197.10
  Batch   600/51579   | Avg Loss (last 200): 2.7170 | PPL:   15.14 | Tokens/Sec: 5762.82


KeyboardInterrupt: 

In [45]:
def translate_sentence(sentence: str, model, tokenizer, device, max_len=50):

    model.eval()

    src_ids = [tokenizer.token_to_id('[SOS]')] + tokenizer.encode(sentence).ids + [tokenizer.token_to_id('[EOS]')]
    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)
    src_mask = (src_tensor != PAD_token).unsqueeze(1).unsqueeze(2)

    with torch.no_grad():
        encoder_output = model.encode(src_tensor, src_mask)

    tgt_tokens = [tokenizer.token_to_id('[SOS]')]

    for _ in range(max_len):
        tgt_tensor = torch.tensor(tgt_tokens).unsqueeze(0).to(device)

        trg_mask_padding = (tgt_tensor != PAD_token).unsqueeze(1).unsqueeze(2)
        subsequent_mask = torch.tril(torch.ones(1, tgt_tensor.size(1), tgt_tensor.size(1))).to(device)
        trg_mask = trg_mask_padding & (subsequent_mask == 1)

        with torch.no_grad():
            decoder_output = model.decode(encoder_output, src_mask, tgt_tensor, trg_mask)
            logits = model.project(decoder_output)

        pred_token = logits.argmax(dim=-1)[0, -1].item()

        tgt_tokens.append(pred_token)

        if pred_token == tokenizer.token_to_id('[EOS]'):
            break

    translated_text = tokenizer.decode(tgt_tokens, skip_special_tokens=True)

    return translated_text

In [47]:

english_sentence_1 = "what is your name?"
hindi_translation_1 = translate_sentence(english_sentence_1, model, tokenizer, device)
print(f"English:  {english_sentence_1}")
print(f"Hindi:    {hindi_translation_1}")
print("-" * 30)


english_sentence_2 = "this is a very important project for our team."
hindi_translation_2 = translate_sentence(english_sentence_2, model, tokenizer, device)
print(f"English:  {english_sentence_2}")
print(f"Hindi:    {hindi_translation_2}")
print("-" * 30)


english_sentence_3 = "the weather is beautiful today."
hindi_translation_3 = translate_sentence(english_sentence_3, model, tokenizer, device)
print(f"English:  {english_sentence_3}")
print(f"Hindi:    {hindi_translation_3}")
print("-" * 30)

English:  what is your name?
Hindi:    आपका नाम क्या है ?
------------------------------
English:  this is a very important project for our team.
Hindi:    यह हमारे टीम के लिए एक बहुत महत्वपूर्ण परियोजना है ।
------------------------------
English:  the weather is beautiful today.
Hindi:    आज मौसम सुंदर है ।
------------------------------


In [53]:
english_sentence_3 = "he is my son. I am her father"
hindi_translation_3 = translate_sentence(english_sentence_3, model, tokenizer, device)
print(f"English:  {english_sentence_3}")
print(f"Hindi:    {hindi_translation_3}")
print("-" * 30)

English:  he is my son. I am her father
Hindi:    वह मेरे बेटे है । मैं उसके पिता हूँ ।
------------------------------
